<a href="https://colab.research.google.com/github/Inaka06/kc-codeqa-reproduce/blob/transformer-ver/CodeQA_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CodeQA versi ringan: Transformer saja + Fine-tuning

Reproduksi ringan paper **CodeQA: A Question Answering Dataset for Source Code Comprehension** (Liu & Wan, Findings of EMNLP 2021).

- **Tugas**: diberi potongan kode + pertanyaan, model menghasilkan jawaban free-form.
- **Model**: satu transformer encoder-decoder pre-trained (**CodeT5-small**, 6 layer, 8 head, hidden 512.
- **Metrik**: BLEU, ROUGE-L, METEOR, EM, F1.

**Sebelum mulai**: `Runtime > Change runtime type > T4 GPU`.

## 1. Install dan cek GPU

In [1]:
!pip -q install transformers sacrebleu nltk sentencepiece

In [2]:
import os, glob, json, math, random, time
from collections import Counter
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 hanya di GPU baru (A100/L4). T4 memakai fp32 karena fp16 pada T5 sering NaN.
BF16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (aktifkan GPU!)", "| bf16:", BF16)

torch 2.11.0+cu130 | device: cuda | Tesla T4 | bf16: False


## 2. Siapkan dataset

Dataset lengkap ada di link Google Drive pada README repo: https://github.com/jadecxliu/CodeQA

Caranya:
1. Download dataset dari link di README, upload ke Google Drive kamu (misalnya `MyDrive/codeqa.zip`).
2. Jalankan sel di bawah untuk mount Drive dan unzip ke `/content/data`.

Untuk uji cepat tanpa dataset lengkap, kamu juga bisa pakai `data_sample` dari repo (sel clone di bawah).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# --- Opsi A: dataset lengkap dari Drive (sesuaikan path zip-nya) ---
ZIP_PATH = "/content/drive/MyDrive/CodeQA/CodeQA.zip"
if os.path.exists(ZIP_PATH):
    !unzip -q -o "$ZIP_PATH" -d /content/data
else:
    print("zip tidak ditemukan -> pakai Opsi B (data_sample dari repo)")
    !git clone -q https://github.com/jadecxliu/CodeQA.git /content/CodeQA

# Lihat struktur file supaya mudah menentukan data_dir
!find /content/data /content/CodeQA -maxdepth 4 -not -path '*/.git*' 2>/dev/null | head -60

/content/data
/content/data/CodeQA_data
/content/data/CodeQA_data/python
/content/data/CodeQA_data/python/dev
/content/data/CodeQA_data/python/dev/dev.question
/content/data/CodeQA_data/python/dev/dev.answer
/content/data/CodeQA_data/python/dev/dev.code
/content/data/CodeQA_data/python/dev/dev.code.original
/content/data/CodeQA_data/python/train
/content/data/CodeQA_data/python/train/train.answer
/content/data/CodeQA_data/python/train/train.code.original
/content/data/CodeQA_data/python/train/train.question
/content/data/CodeQA_data/python/train/train.code
/content/data/CodeQA_data/python/test
/content/data/CodeQA_data/python/test/test.answer
/content/data/CodeQA_data/python/test/test.question
/content/data/CodeQA_data/python/test/test.code.original
/content/data/CodeQA_data/python/test/test.code
/content/data/CodeQA_data/java
/content/data/CodeQA_data/java/dev
/content/data/CodeQA_data/java/dev/dev.question
/content/data/CodeQA_data/java/dev/dev.answer
/content/data/CodeQA_data/java/d

## 3. Konfigurasi

`SMOKE = True` untuk uji cepat (500 data latih, 1 epoch) supaya kamu tahu pipeline jalan dan bisa mengira-ngira kecepatan training. Setelah itu ubah ke `False` untuk training penuh.

In [5]:
from types import SimpleNamespace

SMOKE = True   # True = uji cepat | False = training penuh

cfg = SimpleNamespace(
    data_dir   = "/content/data",      # folder dataset (ganti ke /content/CodeQA kalau pakai data_sample)
    lang       = "python",             # "python" atau "java"
    code_ext   = "code",               # "code" (versi diproses) atau "code.original"
    model      = "Salesforce/codet5-small",   # bisa diganti codet5-base kalau GPU cukup
    max_src    = 384,                  # panjang maksimal input (pertanyaan + kode)
    max_tgt    = 32,                   # panjang maksimal jawaban
    beam       = 4,
    batch      = 16,
    grad_accum = 2,                    # batch efektif = 16 x 2 = 32
    lr         = 5e-5,
    epochs     = 1 if SMOKE else 5,
    patience   = 2,                    # early stopping
    max_train  = 500 if SMOKE else None,    # None = pakai semua data latih
    dev_subset = 200 if SMOKE else 2000,    # data dev untuk validasi tiap epoch
    test_limit = 200 if SMOKE else None,    # None = test set penuh
    eval_bs    = 32,
    out        = "/content/out/codeqa",
    seed       = 42,
)

random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
os.makedirs(cfg.out, exist_ok=True)

## 4. Data

In [6]:
!pip -q install "transformers>=4.40,<5"

In [7]:
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def find_split_dir(data_dir, lang, split):
    # cari <split>.question di mana pun di bawah data_dir (utamakan folder yang memuat nama bahasa)
    hits = [os.path.dirname(h) for h in
            glob.glob(os.path.join(data_dir, "**", f"{split}.question"), recursive=True)]
    hits = [h for h in hits if lang in h.lower()] or hits
    if not hits:
        raise FileNotFoundError(
            f"'{split}.question' tidak ditemukan di {data_dir}. Cek output sel struktur file "
            "di langkah 2 lalu sesuaikan cfg.data_dir atau nama file di CodeQADataset.")
    return sorted(hits)[0]

def build_input(question, code):
    # pertanyaan di depan supaya tidak ikut terpotong saat kode panjang
    return f"question: {question} code: {code}"

class CodeQADataset(Dataset):
    def __init__(self, split, limit=None):
        d = find_split_dir(cfg.data_dir, cfg.lang, split)
        q = read_lines(os.path.join(d, f"{split}.question"))
        a = read_lines(os.path.join(d, f"{split}.answer"))
        c = read_lines(os.path.join(d, f"{split}.{cfg.code_ext}"))
        assert len(q) == len(a) == len(c), "jumlah baris question/answer/code tidak sama"
        idx = list(range(len(q)))
        if limit and limit < len(idx):
            random.Random(cfg.seed).shuffle(idx)
            idx = idx[:limit]
        self.items = [(q[i], c[i], a[i]) for i in idx]
        print(f"{split}: {len(self.items)} contoh dari {d}")
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

tok = AutoTokenizer.from_pretrained(cfg.model)

def make_collate(tok):
    def collate(batch):
        src = [build_input(q, c) for q, c, _ in batch]
        answers = [a for _, _, a in batch]
        enc = tok(src, max_length=cfg.max_src, truncation=True, padding=True, return_tensors="pt")
        lab = tok(text_target=answers, max_length=cfg.max_tgt, truncation=True,
                  padding=True, return_tensors="pt")["input_ids"]
        lab[lab == tok.pad_token_id] = -100
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                "labels": lab, "answers": answers}
    return collate

train_ds = CodeQADataset("train", cfg.max_train)
dev_ds   = CodeQADataset("dev",   cfg.dev_subset)
test_ds  = CodeQADataset("test",  cfg.test_limit)
print("\ncontoh:", train_ds[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train: 500 contoh dari /content/data/CodeQA_data/python/train
dev: 200 contoh dari /content/data/CodeQA_data/python/dev
test: 200 contoh dari /content/data/CodeQA_data/python/test

contoh: ('Where did their types store ?', "def post 2 data func def wrapper self request request DATA request POS Tif '$$original JSON' in request POST request DATA json loads request POST['$$original JSON'] return func self request return wrapper", 'on client - side')


## 5. Metrik

- **BLEU**: seberapa banyak potongan kata jawaban model yang sama dengan jawaban asli.
- **ROUGE-L**: seberapa panjang urutan kata yang sama antara jawaban model dan jawaban asli.
- **METEOR**: seperti BLEU tapi memperhitungkan sinonim dan bentuk kata.
- **EM (Exact Match)**: persentase jawaban yang persis sama dengan jawaban asli.
- **F1**: kecocokan kata per kata antara jawaban model dan jawaban asli (gabungan precision dan recall).

In [8]:
from sacrebleu import corpus_bleu

def norm(s): return " ".join(s.lower().split())

def token_f1(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    if not p or not g: return float(p == g)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

def _lcs(a, b):
    dp = [0] * (len(b) + 1)
    for i in range(1, len(a) + 1):
        prev = 0
        for j in range(1, len(b) + 1):
            tmp = dp[j]
            dp[j] = prev + 1 if a[i-1] == b[j-1] else max(dp[j], dp[j-1])
            prev = tmp
    return dp[-1]

def rouge_l(pred, gold, beta=1.2):
    p, g = pred.split(), gold.split()
    if not p or not g: return 0.0
    l = _lcs(p, g)
    if l == 0: return 0.0
    prec, rec = l / len(p), l / len(g)
    return (1 + beta**2) * prec * rec / (rec + beta**2 * prec)

def meteor_avg(preds, golds):
    try:
        import nltk
        from nltk.translate.meteor_score import meteor_score
        nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
        return 100 * float(np.mean([meteor_score([g.split()], p.split()) for p, g in zip(preds, golds)]))
    except Exception as e:
        print("[warn] METEOR dilewati:", e); return None

def score(preds, golds, with_meteor=False):
    res = {
        "BLEU":    corpus_bleu(preds, [golds]).score,
        "ROUGE-L": 100 * float(np.mean([rouge_l(p, g) for p, g in zip(preds, golds)])),
        "EM":      100 * float(np.mean([norm(p) == norm(g) for p, g in zip(preds, golds)])),
        "F1":      100 * float(np.mean([token_f1(p, g) for p, g in zip(preds, golds)])),
    }
    if with_meteor:
        m = meteor_avg(preds, golds)
        if m is not None: res["METEOR"] = m
    return res

## 6. Fungsi generate dan training

In [9]:
def amp_ctx():
    return torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=BF16)

@torch.no_grad()
def generate(model, tok, ds):
    model.eval()
    loader = DataLoader(ds, batch_size=cfg.eval_bs, shuffle=False, collate_fn=make_collate(tok))
    preds = []
    for b in loader:
        with amp_ctx():
            out = model.generate(input_ids=b["input_ids"].to(device),
                                 attention_mask=b["attention_mask"].to(device),
                                 num_beams=cfg.beam, max_new_tokens=cfg.max_tgt, early_stopping=True)
        preds += [s.strip() for s in tok.batch_decode(out, skip_special_tokens=True)]
    return preds

def train():
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model).to(device)
    print(f"parameter: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
    loader = DataLoader(train_ds, batch_size=cfg.batch, shuffle=True, collate_fn=make_collate(tok))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    total = math.ceil(len(loader) / cfg.grad_accum) * cfg.epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * total), total)

    best_bleu, bad, history = -1.0, 0, []
    best_dir = os.path.join(cfg.out, "best")
    dev_golds = [a for _, _, a in dev_ds.items]

    for epoch in range(1, cfg.epochs + 1):
        model.train(); t0 = time.time(); run = 0.0
        for step, b in enumerate(loader, 1):
            with amp_ctx():
                loss = model(input_ids=b["input_ids"].to(device),
                             attention_mask=b["attention_mask"].to(device),
                             labels=b["labels"].to(device)).loss
            (loss / cfg.grad_accum).backward()
            run += loss.item()
            if step % cfg.grad_accum == 0 or step == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 100 == 0 or step == len(loader):
                print(f"  epoch {epoch} step {step}/{len(loader)} loss={run/step:.4f} "
                      f"({(time.time()-t0)/60:.1f} menit)")

        res = score(generate(model, tok, dev_ds), dev_golds)
        res.update(epoch=epoch, train_loss=run / len(loader), minutes=(time.time() - t0) / 60)
        history.append(res)
        print(f"[epoch {epoch}] " + " | ".join(f"{k}={v:.2f}" for k, v in res.items()))

        if res["BLEU"] > best_bleu:   # checkpoint terbaik dipilih berdasarkan BLEU (seperti paper)
            best_bleu, bad = res["BLEU"], 0
            model.save_pretrained(best_dir); tok.save_pretrained(best_dir)
            print("  -> checkpoint terbaik disimpan")
        else:
            bad += 1
            if bad >= cfg.patience:
                print("early stopping"); break
    return history, best_dir

## 7. Jalankan fine-tuning

In [10]:
import pandas as pd
history, best_dir = train()
pd.DataFrame(history).round(2)

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

parameter: 60.5M


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

  epoch 1 step 32/32 loss=6.0486 (0.2 menit)
[epoch 1] BLEU=0.48 | ROUGE-L=9.98 | EM=0.50 | F1=11.34 | epoch=1.00 | train_loss=6.05 | minutes=0.24
  -> checkpoint terbaik disimpan


,BLEU,ROUGE-L,EM,F1,epoch,train_loss,minutes
0,0.48,9.98,0.5,11.34,1,6.05,0.24


## 8. Evaluasi di test set dan bandingkan dengan paper

Kalau `SMOKE = True`, angkanya belum bermakna (data latih cuma 500). Jalankan training penuh untuk angka yang layak dibandingkan.

In [11]:
best = AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(device)
test_golds = [a for _, _, a in test_ds.items]
test_preds = generate(best, tok, test_ds)
res = score(test_preds, test_golds, with_meteor=True)

# Angka test set dari Tabel 7 dan 8 paper
paper = {
    "java":   {"Paper: Transformer": [30.22, 21.26, 6.77, 4.41, 22.04],
               "Paper: CodeBERT":    [32.40, 28.22, 10.10, 6.20, 29.20]},
    "python": {"Paper: Transformer": [30.69, 23.26, 7.80, 3.35, 24.59],
               "Paper: CodeBERT":    [34.86, 30.28, 12.51, 4.93, 31.56]},
}
cols = ["BLEU", "ROUGE-L", "METEOR", "EM", "F1"]
rows = {f"Versi ringan ({cfg.model.split('/')[-1]})": [res.get(c) for c in cols]}
rows.update(paper[cfg.lang])
table = pd.DataFrame(rows, index=cols).T.round(2)
table

,BLEU,ROUGE-L,METEOR,EM,F1
Versi ringan (codet5-small),0.42,8.65,4.37,0.00,9.69
Paper: Transformer,30.69,23.26,7.80,3.35,24.59
Paper: CodeBERT,34.86,30.28,12.51,4.93,31.56


## 9. Analisis kualitatif dan per jenis pertanyaan

In [12]:
# Contoh prediksi (untuk laporan, seperti Tabel 10-12 di paper)
for i in random.Random(1).sample(range(len(test_ds)), min(10, len(test_ds))):
    q, c, a = test_ds.items[i]
    print("CODE :", c[:150] + ("..." if len(c) > 150 else ""))
    print("Q    :", q)
    print("GOLD :", a)
    print("PRED :", test_preds[i], "\n")

CODE : def hps Model Synth hfreq hmag hphase stoc Env N H fs yh SM sine Model Synth hfreq hmag hphase N H fs yst STM stochastic Model Synth stoc Env H H * 2 ...
Q    : What do a sound use ?
GOLD : the harmonic plus stochastic model hfreq
PRED : sine 

CODE : def test settings clean revert env modified 'outer'env notmodified 'outer'with settings modified 'inner' notmodified 'inner' inner only 'only' clean r...
Q    : What should settings revert only ?
GOLD : values matching input values
PRED : inner only 

CODE : def merge prefix deque size if len deque 1 and len deque[ 0 ] < size returnprefix []remaining sizewhile deque and remaining > 0 chunk deque popleft if...
Q    : What does the code replace with a single string of up to size bytes ?
GOLD : the first entries in a deque of strings
PRED : prefix 

CODE : @CELERY APP taskdef send ccx course published course key course key Course Locator from string course key for ccx in Custom Course For Ed X objects fi...
Q    : What does the code f

In [13]:
# Skor per jenis pertanyaan (kata tanya pertama)
def qtype(q):
    w = q.lower().split()
    if q.lower().startswith("for what purpose"): return "for what purpose"
    first = w[0] if w else ""
    return first if first in {"what","how","where","when","why"} else "yes/no & lain"

df = pd.DataFrame({"type": [qtype(q) for q, _, _ in test_ds.items],
                   "F1": [token_f1(p, g) * 100 for p, g in zip(test_preds, test_golds)],
                   "EM": [float(norm(p) == norm(g)) * 100 for p, g in zip(test_preds, test_golds)]})
df.groupby("type").agg(jumlah=("F1", "size"), F1=("F1", "mean"), EM=("EM", "mean")).round(2)

,jumlah,F1,EM
type,,,
for what purpose,9,4.59,0.0
how,13,3.32,0.0
what,153,11.49,0.0
when,6,5.56,0.0
where,14,4.42,0.0
yes/no & lain,5,0.00,0.0


## 10. Demo: tanya langsung ke kode

In [14]:
def ask(code_str, question):
    code_str = " ".join(code_str.split())   # jadikan satu baris, mirip format dataset
    enc = tok([build_input(question, code_str)], max_length=cfg.max_src, truncation=True, return_tensors="pt")
    with torch.no_grad(), amp_ctx():
        out = best.generate(input_ids=enc["input_ids"].to(device),
                            attention_mask=enc["attention_mask"].to(device),
                            num_beams=cfg.beam, max_new_tokens=cfg.max_tgt)
    return tok.decode(out[0], skip_special_tokens=True).strip()

demo_code = '''
def calc_angle(v1, v2, v3):
    v1 = v1 - v2
    v3 = v3 - v2
    return v1.angle(v3)
'''
print(ask(demo_code, "What does the code calculate?"))

v1.angle(v3)


## 11. Simpan hasil ke Google Drive

In [15]:
with open(os.path.join(cfg.out, "test_results.json"), "w") as f: json.dump(res, f, indent=2)
with open(os.path.join(cfg.out, "test_predictions.jsonl"), "w", encoding="utf-8") as f:
    for (q, _, a), p in zip(test_ds.items, test_preds):
        f.write(json.dumps({"question": q, "gold": a, "pred": p}, ensure_ascii=False) + "\n")

!mkdir -p /content/drive/MyDrive/codeqa_out
!cp -r {cfg.out} /content/drive/MyDrive/codeqa_out/
print("tersimpan di MyDrive/codeqa_out")

tersimpan di MyDrive/codeqa_out


In [16]:
import json, pandas as pd
d = "/content/drive/MyDrive/codeqa_out/codeqa"
print(json.load(open(f"{d}/test_results.json")))
pd.read_json(f"{d}/test_predictions.jsonl", lines=True).head(20)

{'BLEU': 0.41651951629467454, 'ROUGE-L': 8.650873941831145, 'EM': 0.0, 'F1': 9.686347730859925, 'METEOR': 4.365404040135524}


,question,gold,pred
0,What has a fixed prime / generator group ?,c{kexalgorithm,Kex
1,What does the code create if it already exists ?,a metadata - schema object,session
2,What do all of the stochastic nodes in use ?,the rng uniform - > uniform,rng
3,How are the credentials retrieved ?,using get_credentials that checks the environm...,credentials
4,What does the code add ?,attachment_specs table,attachment
5,What does the code execute ?,"a "" start "" action on a vm",server / start
6,What generates the proper documentation ?,the right serializer,class serializer
7,What does the code bring by removing superfluo...,expr nearer to its principal branch,my
8,For what purpose did from the control file cal...,to indicate that fsdev is to be used,FSDEV
9,What does the code get ?,all group snapshots,group


In [17]:
import os, glob, random, torch
from collections import Counter
from google.colab import drive
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

drive.mount('/content/drive')
MODEL_DIR = "/content/drive/MyDrive/codeqa_out/codeqa/best"   # folder "best" from your Drive
DATA_DIR, LANG, CODE_EXT = "/content/data", "python", "code"  # same values as in cfg
N = 30                                                        # max 30 questions

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()

# read the test set
hits = [os.path.dirname(h) for h in glob.glob(f"{DATA_DIR}/**/test.question", recursive=True)]
d = sorted([h for h in hits if LANG in h.lower()] or hits)[0]
rd = lambda ext: [l.rstrip("\n") for l in open(f"{d}/test.{ext}", encoding="utf-8")]
Q, A, C = rd("question"), rd("answer"), rd(CODE_EXT)

norm = lambda s: " ".join(s.lower().split())
def f1(p, g):
    p, g = norm(p).split(), norm(g).split()
    c = sum((Counter(p) & Counter(g)).values())
    if not p or not g or c == 0: return float(p == g)
    pr, rc = c / len(p), c / len(g)
    return 2 * pr * rc / (pr + rc)

em, f1s = [], []
for n, i in enumerate(random.Random(0).sample(range(len(Q)), min(N, 30, len(Q))), 1):
    enc = tok([f"question: {Q[i]} code: {C[i]}"], max_length=384, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(input_ids=enc["input_ids"].to(device),
                             attention_mask=enc["attention_mask"].to(device),
                             num_beams=4, max_new_tokens=32)
    pred = tok.decode(out[0], skip_special_tokens=True).strip()
    em.append(norm(pred) == norm(A[i])); f1s.append(f1(pred, A[i]))
    print(f"[{n}] CODE: {C[i][:120]}...\n    Q   : {Q[i]}\n    GOLD: {A[i]}\n    PRED: {pred}  {'✓' if em[-1] else ''}\n")

print(f"EM = {100*sum(em)/len(em):.1f}%  |  F1 = {100*sum(f1s)/len(f1s):.1f}%  (on {len(em)} questions)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1] CODE: @fixturedef db global dbif db is None db orm new session factory 'sqlite /// memory ' echo True user orm User name getus...
    Q   : What does the code get ?
    GOLD: a db session
    PRED: db  

[2] CODE: def role get name user None host None port None maintenance db None password None runas None return password False all u...
    Q   : How do a dict return ?
    GOLD: with information about users of a postgres server
    PRED: user  

[3] CODE: def register linker name linker if name in predefined linkers raise Value Error ' Linkernamealreadytaken %s' % name pred...
    Q   : How can to be referred ?
    GOLD: by name in mode
    PRED: linker name  

[4] CODE: def open logfile filename mode 'a' filename os path expanduser filename filename os path abspath filename dirname os pat...
    Q   : How does the code open the named log file ?
    GOLD: 